In [ ]:
import pandas as pd

# 1. Load dataset with correct encoding
df = pd.read_csv('train.csv', encoding='latin1')

# 2. Parse Order Date into standardized datetime format
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed', dayfirst=True)

print("Total Records Loaded:", len(df))
df.head(3)

Total Records Loaded: 9800


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,2017-11-08,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96
1,2,CA-2017-152156,2017-11-08,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94
2,3,CA-2017-138688,2017-06-12,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62


In [ ]:
# Calculate total unique orders per customer
customer_orders = df.groupby('Customer ID')['Order ID'].nunique().reset_index()
customer_orders.columns = ['Customer ID', 'Total_Orders']

# Categorize customers as One-time or Repeat customers
customer_orders['Type'] = customer_orders['Total_Orders'].apply(
    lambda x: 'One-time Customer' if x == 1 else 'Repeat Customer'
)

# Display customer type breakdown summary
print("--- CUSTOMER TYPE BREAKDOWN ---")
print(customer_orders['Type'].value_counts())

--- CUSTOMER TYPE BREAKDOWN ---
Type
Repeat Customer      780
One-time Customer     13
Name: count, dtype: int64


In [ ]:
# Calculate Recency, Frequency, and Monetary (RFM) metrics per customer
max_date = df['Order Date'].max()

rfm = df.groupby('Customer ID').agg({
    'Order Date': lambda x: (max_date - x.max()).days, # Recency: Days since last order
    'Order ID': 'nunique',                             # Frequency: Distinct order count
    'Sales': 'sum'                                     # Monetary: Total sales value
}).reset_index()

# Rename columns for clarity and consistency
rfm.columns = ['Customer ID', 'Days_Since_Last_Order', 'Total_Orders', 'Total_Spent']

# Categorize customer status based on inactivity threshold (> 180 days)
rfm['Status'] = rfm['Days_Since_Last_Order'].apply(
    lambda x: 'Churn Risk (Inactive > 6 months)' if x > 180 else 'Active Customer'
)

# Display churn distribution summary
print("--- CHURN STATUS SUMMARY ---")
print(rfm['Status'].value_counts())

# Export processed dataset for Power BI visualization
rfm.to_csv('customer_churn_analysis.csv', index=False)
print("\nExport Success: 'customer_churn_analysis.csv' generated successfully!")

--- CHURN STATUS SUMMARY ---
Status
Active Customer                     591
Churn Risk (Inactive > 6 months)    202
Name: count, dtype: int64

Export Success: 'customer_churn_analysis.csv' generated successfully!


--- YEARLY CUSTOMER & SALES TREND ---
   Order_Year  Total_Customers  Total_Orders  Total_Sales
0        2015              589           947  479856.2081
1        2016              567          1019  459436.0054
2        2017              635          1295  600192.5500
3        2018              690          1661  722052.0192


In [ ]:
# 1. Extract order year from datetime column
df['Order_Year'] = df['Order Date'].dt.year

# 2. Aggregate yearly customer metrics and sales trends
yearly_churn = df.groupby('Order_Year').agg(
    Total_Customers=('Customer ID', 'nunique'),
    Total_Orders=('Order ID', 'nunique'),
    Total_Sales=('Sales', 'sum')
).reset_index()

# Display yearly metrics summary
print("--- YEARLY CUSTOMER & SALES TREND ---")
print(yearly_churn)

# Export yearly trend metrics to CSV
yearly_churn.to_csv('yearly_churn_trend.csv', index=False)
print("\nExport Success: 'yearly_churn_trend.csv' generated successfully!")

--- YEARLY CUSTOMER & SALES TREND ---
   Order_Year  Total_Customers  Total_Orders  Total_Sales
0        2015              589           947  479856.2081
1        2016              567          1019  459436.0054
2        2017              635          1295  600192.5500
3        2018              690          1661  722052.0192

Export Success: 'yearly_churn_trend.csv' generated successfully!


In [ ]:
# 1. Calculate Shipping Delay (in Days)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='mixed', dayfirst=True)
df['Shipping_Delay_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# 2. Categorize Delivery Performance
df['Delivery_Status'] = df['Shipping_Delay_Days'].apply(
    lambda x: 'Delayed (>5 Days)' if x > 5 else 'On-Time / Fast'
)

# 3. Summary Performance by Ship Mode
shipping_summary = df.groupby('Ship Mode').agg(
    Average_Delay_Days=('Shipping_Delay_Days', 'mean'),
    Total_Orders=('Order ID', 'nunique')
).reset_index()

print("--- SHIPPING PERFORMANCE SUMMARY ---")
print(shipping_summary)

# 4. Save CSV for Power BI
df.to_csv('shipping_analysis.csv', index=False)
print("\nExport Success: 'shipping_analysis.csv' created!")

--- SHIPPING PERFORMANCE SUMMARY ---
        Ship Mode  Average_Delay_Days  Total_Orders
0     First Class            2.179214           772
1        Same Day            0.044610           261
2    Second Class            3.249211           944
3  Standard Class            5.008363          2945

Export Success: 'shipping_analysis.csv' created!
